In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..','..')))
from ai_tools import LLMQuery
import gradio as gr # oh yeah!
import json
from scraper import fetch_website_contents, fetch_website_links

In [2]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages. Distinguish between irrelevant and relevant links for the brochure.
Include relative links to your response. Expand relative links to absolute links.
You must respond in strict JSON without any additional text as in this example:

{
    "relevant": {
        "links": [
            {"type": "about page", "url": "https://full.url/goes/here/about"},
            {"type": "careers page", "url": "https://another.full.url/careers"}
        ]
    },
    "irrelevant": {
        "links": [
            {"type": "privacy policy", "url": "https://full.url/goes/here/privacy-policy"},
            {"type": "foreign link not closly related", "url": "https://another.url/with/no/relevance"}
            
        ]
    }
}
"""

link_user_prompt = """
You are provided with a list of links found on a webpage. Filter out irrelevant links and return the relevant links in strict JSON format.
"""


brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:50_000] # Truncate if more than 5,000 characters
    return user_prompt



In [3]:
def get_relevant_links_of_webpage(url):
    link_llm = LLMQuery(system_prompt=link_system_prompt, model="gemini-flash-lite-latest")
    links = fetch_website_links(url)
    relevant_links = link_llm.query(user_prompt=link_user_prompt + "\n\n" + "\n".join(links), json_format=True)
    relevant_links_json = json.loads(relevant_links)
    links_only = {l['url'] for l in relevant_links_json['relevant']['links']}
    links_only.add(url)
    return links_only
    

In [10]:
def fetch_page_and_all_relevant_links(url):
    links = get_relevant_links_of_webpage(url)
    print(links)
    result = ""
    for link in links:
        result += f"\n\n### Link:\n"
        try:
            result += fetch_website_contents(link)
        except Exception as e:
            result += f"Error fetching {link}: {str(e)}"
    return result

In [8]:
def create_brochure(url):
    company_name = "url"
    try: 
        content = fetch_website_contents(url)
        if not content:
            raise ValueError("No content found")
    except Exception as e:
        yield str(e)
        
    brochure_client = LLMQuery(system_prompt=brochure_system_prompt, model="gemini-3-pro-preview")
    output_stream = brochure_client.query_stream(user_prompt=get_brochure_user_prompt(company_name, url), return_generator=True)
    yield from output_stream

In [6]:
#a = create_brochure("Gemeinde Kalchreuth", "https://kalchreuth.de/")

In [11]:
message_input = gr.Textbox(label="Your url:", info="Enter a valid url!", lines=1)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=create_brochure,
    title="Create Brochure", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "https://kalchreuth.de/",
        "https://roeckenhof.de/",
        ], 
    flagging_mode="never"
    )

view.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


{'https://www.heroldsberg.de/wirtschaft-und-gewerbe/industrie/', 'https://www.heroldsberg.de/', 'https://www.heroldsberg.de/gemeindliche-einrichtungen/agenda21/', 'https://www.heroldsberg.de/markt-heroldsberg/ortsgeschichte/', 'https://www.heroldsberg.de/vereine-und-kirchen/vereins-bus/', 'https://www.heroldsberg.de/gesundheit-und-soziales/familienangebote/', 'https://www.heroldsberg.de/service/wichtige-rufnummern/', 'https://www.heroldsberg.de/markt-heroldsberg/rathaus-mitarbeiter-fachbereiche/', 'https://www.heroldsberg.de/wirtschaft-und-gewerbe/bodenrichtwerte/', 'https://www.heroldsberg.de/auszeichnungen/', 'https://www.heroldsberg.de/wp-content/uploads/2025/01/2025-Familienwegweiser.pdf', 'https://www.heroldsberg.de/wp-content/uploads/2025/01/2025-Organigramm.pdf', 'https://www.heroldsberg.de/markt-heroldsberg/daten-und-fakten/', 'https://www.heroldsberg.de/stellenangebote/', 'https://www.heroldsberg.de/kontakt/', 'https://www.heroldsberg.de/antraege-und-formulare/', 'https://www.

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
